# Mean-Reversion BB+RSI Retest Sweep

Runs the MR BB+RSI pipeline on a user-specified `RETEST_PAIRS` list
and ranks results across pairs.


In [ ]:
import sys, os, subprocess, time, logging
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import optuna
import pmm_lab

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")
print(f"Strategy: mean_reversion_bb_rsi (retest)")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")


## 1. Configuration


In [ ]:
# ==============================================================
# MR BB+RSI RETEST SWEEP CONFIGURATION
# ==============================================================
# Retest pattern: optimize a user-provided finalist list rather than
# discovering all pairs. Cross-pair ranking is added in cells 13-14.
# ==============================================================

RETEST_PAIRS = [
    ("XMR-USDT", "nonkyc"),
    ("XMR-USDT", "mexc"),
    # ADD MORE PAIRS AS FINALISTS EMERGE
]
CONNECTORS = sorted(set(connector for _, connector in RETEST_PAIRS))

QUOTE_ASSET = "*"
N_TRIALS = 500
PERC_TRIALS_TEST = 0.05
TOP_N = 75
MIN_ROBUST_SCORE = -5.0
N_JOBS = 8

CONNECTOR_INTERVALS = {"nonkyc": "5m", "mexc": "5m"}
DEFAULT_INTERVAL = "5m"

MIN_DATA_DAYS = 56
MAX_STALE_DAYS = 7
MAX_TRAINING_DAYS = 180

SEARCH_CONTROLLER_COMPAT = False
VALIDATION_CONTROLLER_COMPAT = True
PHASE2_CONTROLLER_COMPAT = True

REFRESH_CLOSE_MODE = "market_close"
INITIAL_BASE_BALANCE = 0.0

TAKER_PROBABILITY_BY_CONNECTOR = {"nonkyc": 0.10, "mexc": 0.0}
DEFAULT_TAKER_PROBABILITY = 0.0

MIN_PHASE1_BEST_FOR_STRESS = -0.5
OBJECTIVE_VERSION = 2

RECENT_BLOCKING_WINDOW_DAYS = 28
RECENT_INFORMATIONAL_WINDOW_DAYS = [14, 7]
RECENT_REPORT_WINDOW_DAYS = sorted(
    dict.fromkeys([RECENT_BLOCKING_WINDOW_DAYS] + RECENT_INFORMATIONAL_WINDOW_DAYS),
    reverse=True,
)

CONNECTORS = [c.strip().lower() for c in CONNECTORS]
INTERVALS_BY_CONNECTOR = {c: CONNECTOR_INTERVALS.get(c, DEFAULT_INTERVAL) for c in CONNECTORS}

from pmm_lab.config.defaults import INTERVAL_SECONDS

print(f"Strategy       : mean_reversion_bb_rsi_v1 (retest)")
print(f"Retest pairs   : {RETEST_PAIRS}")
print(f"Connectors     : {', '.join(CONNECTORS)}")
print(f"Trials/pair    : {N_TRIALS}")
print(f"Top-N stress   : {TOP_N}")
print(f"Max training   : {MAX_TRAINING_DAYS}d" if MAX_TRAINING_DAYS else "Max training   : unlimited")


In [ ]:
# ── Preflight: validate storage + worker configuration ──
from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_postgres = "postgresql" in str(_storage_url).lower()

print(f"Requested N_JOBS: {N_JOBS}")
print(f"Storage backend : {'PostgreSQL' if _is_postgres else 'SQLite (fallback)'}")
print(f"Dispatch mode   : {'process-parallel' if N_JOBS > 1 and _is_postgres else 'serial'}")
if N_JOBS > 1 and not _is_postgres:
    print("WARNING: N_JOBS>1 with SQLite — forcing serial. Set OPTUNA_STORAGE for parallelism.")


## 2. Load Requested Pairs

In [ ]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles
from datetime import datetime, timezone

loader = MongoCandleLoader()

candidates = []
for pair, connector in RETEST_PAIRS:
    interval = INTERVALS_BY_CONNECTOR.get(connector, DEFAULT_INTERVAL)
    candidates.append({
        "connector": connector, "trading_pair": pair, "interval": interval,
    })

print(f"\n{'='*60}")
print(f"Retest sweep: {len(candidates)} pair(s)")
print(f"{'='*60}")
for c in candidates:
    print(f"  {c['connector']:8s} {c['trading_pair']:15s} {c['interval']}")


## 3. Sweep: Optimize Each Connector / Pair

Same pipeline as the multi-exchange sweep, run per-pair from the
`RETEST_PAIRS` list.


In [ ]:
# ── Config guard: ensure configuration cell was executed ──
_required_config = [
    "VALIDATION_CONTROLLER_COMPAT", "SEARCH_CONTROLLER_COMPAT", "PHASE2_CONTROLLER_COMPAT",
    "OBJECTIVE_VERSION", "N_TRIALS", "TOP_N", "MIN_ROBUST_SCORE",
    "N_JOBS", "MIN_PHASE1_BEST_FOR_STRESS",
]
_missing = [v for v in _required_config if v not in globals()]
if _missing:
    import warnings as _w
    _w.warn(
        f"Configuration cell may not have been executed. "
        f"Missing: {', '.join(_missing)}. "
        f"Applying safe defaults — re-run all cells from the top for your custom settings.",
        stacklevel=1,
    )
    if "VALIDATION_CONTROLLER_COMPAT" not in globals():
        VALIDATION_CONTROLLER_COMPAT = True
    if "SEARCH_CONTROLLER_COMPAT" not in globals():
        SEARCH_CONTROLLER_COMPAT = False
    if "PHASE2_CONTROLLER_COMPAT" not in globals():
        PHASE2_CONTROLLER_COMPAT = True
    if "OBJECTIVE_VERSION" not in globals():
        OBJECTIVE_VERSION = 2
    if "N_TRIALS" not in globals():
        N_TRIALS = 200
    if "TOP_N" not in globals():
        TOP_N = 25
    if "MIN_ROBUST_SCORE" not in globals():
        MIN_ROBUST_SCORE = 0.0
    if "N_JOBS" not in globals():
        N_JOBS = 1
    if "MIN_PHASE1_BEST_FOR_STRESS" not in globals():
        MIN_PHASE1_BEST_FOR_STRESS = 0.0

if "REFRESH_CLOSE_MODE" not in globals():
    REFRESH_CLOSE_MODE = "keep"
if "INITIAL_BASE_BALANCE" not in globals():
    INITIAL_BASE_BALANCE = 0.0

if "RECENT_BLOCKING_WINDOW_DAYS" not in globals():
    RECENT_BLOCKING_WINDOW_DAYS = 28
if "RECENT_INFORMATIONAL_WINDOW_DAYS" not in globals():
    RECENT_INFORMATIONAL_WINDOW_DAYS = [14, 7]
if "RECENT_REPORT_WINDOW_DAYS" not in globals():
    RECENT_REPORT_WINDOW_DAYS = sorted(
        dict.fromkeys([RECENT_BLOCKING_WINDOW_DAYS] + RECENT_INFORMATIONAL_WINDOW_DAYS),
        reverse=True,
    )

import os, time
from dataclasses import replace as _replace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import optuna
from tqdm.auto import tqdm

from pmm_lab.data.candles import validate_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.optuna.notebook_dispatch import optimize_study_for_notebook
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.callbacks import (
    DegeneracyCheckCallback, TrialLoggingCallback, TqdmProgressCallback,
)
# MR-specific imports (aliased to match PMM naming) — substitution per prompt 5A
from pmm_lab.optuna.canonicalizer_mean_reversion_bb_rsi import (
    canonicalize_mr_bb_rsi_params as canonicalize_params,
)
from pmm_lab.export.hb_yaml_mr_bb_rsi import (
    export_mr_bb_rsi_yaml as export_yaml,
    MRBBRSIExportParams as ExportParams,
    validate_export_mr_bb_rsi as validate_yaml_file,
)
from pmm_lab.objective.stress import load_stress_scenarios
from pmm_lab.objective.stress_selection import select_best_stressed_candidate
from pmm_lab.objective.walkforward_dispatch import run_walk_forward_dispatch
from pmm_lab.objective.objective import REJECT_SCORE, objective_v1
from pmm_lab.report.report_md import generate_report, run_stop_ship_checks
from pmm_lab.objective.recent_window import evaluate_recent_window
from pmm_lab.objective.holdout import evaluate_holdout, split_holdout
from pmm_lab.objective.dataset_split import split_for_release_gate
from pmm_lab.objective.signal_cache import SharedSignalCache
from pmm_lab.optuna.sensitivity import compute_sensitivity, MR_PERTURBABLE_PARAMS
from pmm_lab.optuna.clustering import analyze_top_k
from pmm_lab.parity.feature_parity import check_feature_parity_frozen_mr
from pmm_lab.parity.fixtures import load_frozen_fixture
from pmm_lab.data.candles import hash_candles

# Preload stress scenarios once
stress_scenarios = load_stress_scenarios()

rules_db = load_exchange_rules()
sweep_results = []
sweep_start = time.time()

_pair_bar = tqdm(
    total=len(candidates), position=0, leave=True, desc="Pairs",
)

for pair_idx, pair_info in enumerate(candidates):
    connector = pair_info["connector"]
    pair = pair_info["trading_pair"]
    interval = pair_info["interval"]
    bar_interval_seconds = INTERVAL_SECONDS[interval]
    _pair_bar.set_postfix_str(f"{connector}/{pair}")

    print(f"\n{'='*60}")
    print(f"  [{pair_idx+1}/{len(candidates)}] {connector} / {pair} / {interval}")
    print(f"{'='*60}")

    pair_start = time.time()

    # ── Load candles ──
    try:
        _start_ts = int(pair_info["first_ts"]) if (MAX_TRAINING_DAYS is not None and "first_ts" in pair_info) else None
        query = DataQuery(connector=connector, trading_pair=pair, interval=interval, start_ts=_start_ts)
        candles = loader.load_range(query)
        audit = validate_candles(candles, interval=interval, strict=True)
        if not audit.passed_strict:
            print(f"  SKIP: audit failed — {audit.failure_reasons}")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "audit_fail", "robust_score": None})
            _pair_bar.update(1)
            continue
        dataset_hash = hash_candles(candles)
    except Exception as e:
        print(f"  SKIP: load failed — {e}")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "load_fail", "robust_score": None})
        _pair_bar.update(1)
        continue

    # ── Dataset split for release gate (informational) ──
    try:
        dataset_slices = split_for_release_gate(
            candles, recent_days=RECENT_BLOCKING_WINDOW_DAYS, holdout_fraction=0.20,
            min_pre_release_bars=200, min_holdout_bars=50,
        )
        dev_candles = dataset_slices.dev_candles
        dev_dataset_hash = hash_candles(dev_candles)
        print(f"  Split: dev={len(dev_candles)} holdout={len(dataset_slices.holdout_candles)} recent={len(dataset_slices.recent_release_candles)}")
    except ValueError as e:
        print(f"  Split failed ({e}), using full candles")
        dataset_slices = None
        dev_candles = candles
        dev_dataset_hash = dataset_hash

    # ── Exchange rules ──
    try:
        pair_rules = resolve_pair_rules(rules_db, connector, pair)
    except KeyError:
        try:
            pair_rules = resolve_pair_rules(rules_db, connector, "DEFAULT")
        except KeyError:
            print(f"  SKIP: no exchange rules for {connector}/{pair}")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "no_rules", "robust_score": None})
            _pair_bar.update(1)
            continue

    taker_prob = TAKER_PROBABILITY_BY_CONNECTOR.get(connector, DEFAULT_TAKER_PROBABILITY)
    ref_price = float(np.median(candles["close"]))

    # ── Auto-scale walk-forward windows ──
    dataset_days = len(candles) * bar_interval_seconds / 86400
    if dataset_days >= 120:
        train_days, test_days, step_days = 42.0, 14.0, 14.0
    elif dataset_days >= 60:
        train_days, test_days, step_days = 21.0, 7.0, 7.0
    elif dataset_days >= 28:
        train_days, test_days, step_days = 10.0, 4.0, 4.0
    else:
        print(f"  SKIP: only {dataset_days:.1f} days of data")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "insufficient_data", "robust_score": None})
        _pair_bar.update(1)
        continue

    print(f"  Candles: {len(candles):,}  Days: {dataset_days:.1f}  "
          f"WF: {train_days}/{test_days}/{step_days}d  Ref: {ref_price:,.4f}")

    # ── Phase 1: Optimization with inner tqdm bar + DegeneracyCheck ──
    study_name = f"{connector}_{pair}_{interval}_mr_bb_rsi_v1"

    _trial_bar = tqdm(total=N_TRIALS, position=1, leave=False, desc="trials")
    _trial_cb = TqdmProgressCallback(_trial_bar, show_best=True)

    try:
        study = optimize_study_for_notebook(
            study_name=study_name,
            storage_url=OPTUNA_STORAGE if "OPTUNA_STORAGE" in globals() and OPTUNA_STORAGE else None,
            n_trials=N_TRIALS,
            n_jobs=N_JOBS,
            objective_factory=create_objective,
            factory_kwargs=dict(
                candles=dev_candles,
                pair_rules=pair_rules,
                bar_interval_seconds=bar_interval_seconds,
                dataset_hash=dev_dataset_hash,
                reference_price=ref_price,
                strategy_name="mean_reversion_bb_rsi",
                train_days=train_days,
                test_days=test_days,
                step_days=step_days,
                run_stress=False,
                controller_compat=SEARCH_CONTROLLER_COMPAT,
                objective_version=OBJECTIVE_VERSION,
                refresh_close_mode=REFRESH_CLOSE_MODE,
                initial_base_balance=INITIAL_BASE_BALANCE,
                taker_probability=taker_prob,
            ),
            callbacks=[DegeneracyCheckCallback(), _trial_cb],
            n_startup_trials=int(N_TRIALS * PERC_TRIALS_TEST) if "PERC_TRIALS_TEST" in globals() else 15,
        )

        completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
        ranked = sorted(
            [t for t in completed if t.value is not None],
            key=lambda t: t.value, reverse=True,
        )

        if not ranked:
            print(f"  Phase 1: {len(completed)} complete, {len(pruned)} pruned — NO COMPLETED TRIALS")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "no_completed_trials", "robust_score": None})
            _trial_bar.close()
            _pair_bar.update(1)
            continue

        best_val = ranked[0].value
        print(f"  Phase 1: {len(completed)} complete, {len(pruned)} pruned, best={best_val:.4f}")
    except Exception as e:
        print(f"  SKIP: optimization failed — {e}")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "optim_fail", "robust_score": None})
        _trial_bar.close()
        _pair_bar.update(1)
        continue
    finally:
        try:
            _trial_bar.close()
        except Exception:
            pass

    # ── Phase 1 score gate (informational — log and continue to Phase 2) ──
    phase1_below_threshold = best_val <= MIN_PHASE1_BEST_FOR_STRESS
    if phase1_below_threshold:
        print(f"  INFO: phase-1 best ({best_val:.4f}) <= {MIN_PHASE1_BEST_FOR_STRESS}; continuing anyway")

    phase1_pair_elapsed = time.time() - pair_start
    print(f"  Phase 1 time: ({phase1_pair_elapsed/60:.1f}min)")

    # ── Phase 2: Stress top N (dedup via to_fingerprint, MR-local stress) ──
    try:
        top_trials = ranked[:min(TOP_N, len(ranked))]
        top_candidates = []
        for trial in top_trials:
            raw = dict(trial.params)
            raw.setdefault("min_trend_slope", 0.0)
            raw.setdefault("max_spread_pct", 0.006)
            raw.setdefault("max_trades_per_day", 6)
            raw.setdefault("max_executors_per_side", 1)
            raw.setdefault("total_amount_quote", 300.0)
            bundle, reject = canonicalize_params(
                raw, pair_rules, ref_price, bar_interval_seconds=bar_interval_seconds,
            )
            if bundle is not None:
                sc = _replace(bundle.strategy_config, controller_compat=PHASE2_CONTROLLER_COMPAT)
                ec = _replace(bundle.engine_config, taker_probability=taker_prob)
                top_candidates.append({
                    "trial_number": trial.number,
                    "phase1_score": trial.value,
                    "params": trial.params,
                    "config": sc,
                    "engine_config": ec,
                })

        if not top_candidates:
            print(f"  SKIP: no valid configs to stress test")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "no_valid_configs", "robust_score": None})
            _pair_bar.update(1)
            continue

        # Dedup by full config fingerprint
        seen_configs = {}
        deduped_candidates = []
        for candidate in top_candidates:
            fingerprint = candidate["config"].to_fingerprint()
            if fingerprint not in seen_configs:
                seen_configs[fingerprint] = True
                deduped_candidates.append(candidate)
        print(f"  Phase 2: controller_compat={PHASE2_CONTROLLER_COMPAT} (search={SEARCH_CONTROLLER_COMPAT})")
        print(f"  Deduped: {len(top_candidates)} -> {len(deduped_candidates)} unique configs")
        top_candidates = deduped_candidates

        # Signal cache for MR: precompute via SharedSignalCache for each unique config
        _shared_cache = SharedSignalCache()
        for cand in top_candidates:
            _shared_cache.get_or_compute(cand["config"], "dev", dev_candles, pair_rules)

        # MR apply_scenario: modify engine_config (not strategy_config)
        from pmm_lab.objective.stress_mean_reversion_bb_rsi import _apply_scenario as _mr_apply_scenario
        def _apply_scenario_fn(strategy_cfg, engine_cfg, pair_rules, scenario):
            new_engine, new_rules = _mr_apply_scenario(engine_cfg, pair_rules, scenario)
            return strategy_cfg, new_engine, new_rules

        best, diag = select_best_stressed_candidate(
            top_candidates, dev_candles, pair_rules, bar_interval_seconds,
            scenarios=stress_scenarios,
            objective_version=OBJECTIVE_VERSION,
            shared_signal_cache=_shared_cache,
            dataset_key="dev",
            apply_scenario_fn=_apply_scenario_fn,
        )

        if best is None:
            print(f"  SKIP: no candidates survived stress testing")
            sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                                  "status": "stress_fail", "robust_score": None})
            _pair_bar.update(1)
            continue

        best_config = best["config"]
        best_engine_config = best["engine_config"]
        best_stress = best["stress_report"]
        bm = best_stress.baseline_metrics

        pair_elapsed = time.time() - pair_start
        print(f"  Best: trial {best['trial_number']}  robust={best['robust_score']:.4f}  "
              f"PnL={bm.pnl_pct:.2f}%  trades={bm.trade_count}  ({pair_elapsed/60:.1f}min)")
        print(f"  Stress diag: evaluated={diag['candidates_evaluated']} "
              f"pruned={diag['candidates_pruned']} "
              f"cache_hits={diag['signal_cache_hits']} misses={diag['signal_cache_misses']}")

    except Exception as e:
        print(f"  SKIP: stress testing failed — {e}")
        sweep_results.append({"connector": connector, "pair": pair, "interval": interval,
                              "status": "stress_fail", "robust_score": None, "error": str(e)})
        _pair_bar.update(1)
        continue

    # ── Finalist validation ──
    val_config = _replace(best_config, controller_compat=VALIDATION_CONTROLLER_COMPAT)
    val_engine = _replace(
        best_engine_config,
        refresh_close_mode=REFRESH_CLOSE_MODE,
        initial_base_balance=INITIAL_BASE_BALANCE,
        taker_probability=taker_prob,
    )

    recent_window_results = {}
    _shared_cache_full = SharedSignalCache()
    _recent_signals = _shared_cache_full.get_or_compute(
        val_config, "full", candles, pair_rules,
    )

    for _rw_days in RECENT_REPORT_WINDOW_DAYS:
        try:
            _rw = evaluate_recent_window(
                full_candles=candles, config=val_config, pair_rules=pair_rules,
                bar_interval_seconds=bar_interval_seconds,
                recent_days=_rw_days, run_stress=False,
                objective_version=OBJECTIVE_VERSION,
                precomputed_signals=_recent_signals,
                shared_signal_cache=_shared_cache_full,
                engine_config=val_engine,
            )
            recent_window_results[_rw_days] = _rw
            _role = "BLOCKER" if _rw_days == RECENT_BLOCKING_WINDOW_DAYS else "INFO"
            print(f"  Recent {_rw_days}d [{_role}]: {'PASS' if _rw.passed else 'FAIL'} — {_rw.reason}")
        except Exception as e:
            _role = "BLOCKER" if _rw_days == RECENT_BLOCKING_WINDOW_DAYS else "INFO"
            print(f"  Recent {_rw_days}d [{_role}]: ERROR — {e}")

    recent_window_result = recent_window_results.get(RECENT_BLOCKING_WINDOW_DAYS)

    holdout_report = None
    try:
        if dataset_slices is not None:
            holdout_candles_h = dataset_slices.holdout_candles
            holdout_start_idx = dataset_slices.holdout_start_idx_in_pre_release
        else:
            dev_candles_h, holdout_candles_h = split_holdout(candles, 0.20, min_holdout_bars=50)
            holdout_start_idx = len(dev_candles_h)
        holdout_candidates = [(val_config, best.get("robust_score", 0.0))]
        for t_idx in range(1, min(5, len(top_candidates))):
            tc = top_candidates[t_idx]
            tc_bundle, _ = canonicalize_params(
                dict(tc["params"], min_trend_slope=0.0, max_spread_pct=0.006,
                     max_trades_per_day=6, max_executors_per_side=1, total_amount_quote=300.0),
                pair_rules, ref_price, bar_interval_seconds=bar_interval_seconds,
            )
            if tc_bundle is not None:
                tc_cfg = _replace(tc_bundle.strategy_config, controller_compat=VALIDATION_CONTROLLER_COMPAT)
                holdout_candidates.append((tc_cfg, tc.get("phase1_score", 0.0)))
        holdout_report = evaluate_holdout(
            holdout_candles_h, holdout_candidates, pair_rules, bar_interval_seconds,
            run_stress=False, objective_version=OBJECTIVE_VERSION,
            full_candles=candles, holdout_start_idx=holdout_start_idx,
            shared_signal_cache=_shared_cache_full,
            engine_config=val_engine,
        )
        print(f"  Holdout: {'PASS' if holdout_report.exported_holdout_passed else 'FAIL'}")
    except Exception as e:
        print(f"  Holdout: ERROR — {e}")

    sensitivity_report = None
    sensitivity_penalty = None
    try:
        def _mr_canon_adapter(params, pair_rules_arg, ref_price_arg, **kwargs):
            raw = dict(params)
            raw.setdefault("min_trend_slope", 0.0)
            raw.setdefault("max_spread_pct", 0.006)
            raw.setdefault("max_trades_per_day", 6)
            raw.setdefault("max_executors_per_side", 1)
            raw.setdefault("total_amount_quote", 300.0)
            return canonicalize_params(
                raw, pair_rules_arg, ref_price_arg, bar_interval_seconds=bar_interval_seconds,
            )
        sensitivity_report = compute_sensitivity(
            best["params"], candles, pair_rules, bar_interval_seconds, ref_price,
            objective_version=OBJECTIVE_VERSION,
            controller_compat=VALIDATION_CONTROLLER_COMPAT,
            shared_signal_cache=_shared_cache_full,
            canonicalize_fn=_mr_canon_adapter,
            perturb_params=MR_PERTURBABLE_PARAMS,
        )
        sensitivity_penalty = sensitivity_report.sensitivity_penalty
        print(f"  Sensitivity: penalty={sensitivity_penalty:.4f}")
    except Exception as e:
        print(f"  Sensitivity: ERROR — {e}")

    cluster_report = None
    try:
        cluster_report = analyze_top_k(study, k=min(10, len(ranked)))
        print(f"  Clustering: {'CLUSTERED' if cluster_report.is_clustered else 'SCATTERED'}")
    except Exception as e:
        print(f"  Clustering: ERROR — {e}")

    parity_result = None
    long_parity_result = None
    try:
        # Directional MR parity — uses MR-specific fixture and check (P2.3)
        _fix_base = Path("fixtures")
        if _fix_base.is_dir():
            _short = _fix_base / "mr_short_100bar"
            if _short.is_dir():
                _f = load_frozen_fixture(str(_short))
                parity_result = check_feature_parity_frozen_mr(
                    _f.candles, _f.expected_features, _f.config_params,
                )
        print(f"  Parity: short={'PASS' if parity_result and parity_result.passed else 'N/A'}")
    except Exception as e:
        print(f"  Parity: ERROR — {e}")

    full_validation_executed = all([recent_window_result is not None, holdout_report is not None])

    # ── Record result ──
    best_metrics = bm
    best_obj = best_stress.baseline_objective
    # Pull rejection fraction from best trial's user attrs (ML-DIR-007)
    _best_trial_obj = next(
        (t for t in study.trials if t.number == best["trial_number"]), None,
    )
    _reject_frac = None
    if _best_trial_obj is not None:
        _reject_frac = _best_trial_obj.user_attrs.get("total_reject_fraction")
        if _reject_frac is None:
            _reject_frac = _best_trial_obj.user_attrs.get("max_trades_per_day_binding_fraction")
    result_entry = {
        "connector": connector,
        "pair": pair,
        "interval": interval,
        "status": "complete",
        "robust_score": best["robust_score"],
        "baseline_score": best["baseline_score"],
        "worst_score": best["worst_score"],
        "worst_scenario": best["worst_scenario"],
        "pnl_pct": bm.pnl_pct,
        "sharpe": bm.sharpe,
        "max_dd_pct": bm.max_drawdown_pct,
        "trade_count": bm.trade_count,
        "total_fees": bm.total_fees_quote,
        "profit_factor": bm.profit_factor,
        "total_reject_fraction": _reject_frac,
        "trial_number": best["trial_number"],
        "best_config": best_config,
        "best_engine_config": best_engine_config,
        "best_params": best["params"],
        "best_stress": best_stress,
        "dataset_hash": dataset_hash,
        "n_candles": len(candles),
        "dataset_days": dataset_days,
        "train_days": train_days,
        "test_days": test_days,
        "step_days": step_days,
        "study_name": study_name,
        "recent_window_result": recent_window_result,
        "recent_window_results": recent_window_results,
        "holdout_report": holdout_report,
        "sensitivity_report": sensitivity_report,
        "sensitivity_penalty": sensitivity_penalty,
        "cluster_report": cluster_report,
        "parity_result": parity_result,
        "long_parity_result": long_parity_result,
        "full_validation_executed": full_validation_executed,
        "phase1_below_threshold": phase1_below_threshold,
    }
    sweep_results.append(result_entry)

    # ── Validation state machine: fail-closed YAML placement (ML-DIR-001) ──
    # Status values: optimized_only, validation_error, validated_fail, validated_pass
    MANDATORY_GATES = {
        "dataset_audit", "runtime_sanity", "objective_not_degenerate",
        "stress_not_collapsed", "yaml_validates",
        "walkforward_robust", "walkforward_positive_majority",
        "holdout_passed", "holdout_no_collapse",
        "sensitivity_stable", "recent_28d_passed", "top_k_clustered",
    }
    # Once MR + EMA frozen fixtures exist and check_feature_parity_frozen_mr/_ema are green,
    # promote "frozen_parity" into MANDATORY_GATES. Flip this to "mandatory" after P2.3
    # fixtures are committed AND verified to pass on the current feature impls.
    FROZEN_PARITY_POLICY = "advisory"
    if FROZEN_PARITY_POLICY == "mandatory":
        MANDATORY_GATES = MANDATORY_GATES | {"frozen_parity"}

    validation_status = "optimized_only"
    validation_errors = []
    mandatory_gates_failed = []
    yaml_path = None
    checks = {}
    validation_result = None
    wf_result = None

    # Export YAML to .pending/ first; final placement depends on outcome
    try:
        export_params = ExportParams(
            connector_name=connector, trading_pair=pair, interval=interval,
        )
        _out_dir = Path(f"artifacts/direction-custom/mr_bb_rsi/{connector}")
        _out_dir.mkdir(parents=True, exist_ok=True)
        _yaml_filename = f"{connector}_{pair.replace('-', '_').lower()}_{interval}_screening_best.yml"
        _pending_dir = _out_dir / ".pending"
        _pending_dir.mkdir(parents=True, exist_ok=True)
        pending_yaml_path = str(_pending_dir / _yaml_filename)
        export_yaml(best_config, best_engine_config, export_params, Path(pending_yaml_path))
        validation_result = validate_yaml_file(Path(pending_yaml_path))
    except Exception as e:
        validation_errors.append(("export", type(e).__name__, str(e)))
        print(f"  Export/validate error: {e}")

    try:
        wf_result = run_walk_forward_dispatch(
            candles=candles, config=val_config, pair_rules=pair_rules,
            bar_interval_seconds=bar_interval_seconds, dataset_hash=dataset_hash,
            train_days=train_days, test_days=test_days, step_days=step_days,
            objective_version=OBJECTIVE_VERSION,
            engine_config=val_engine,
            shared_signal_cache=_shared_cache_full,
            dataset_key="dev",
        )
        print(f"  Walk-forward: {len(wf_result.folds)} folds, aggregate={wf_result.aggregate_score:.4f}")
    except Exception as e:
        validation_errors.append(("walkforward", type(e).__name__, str(e)))
        print(f"  Walk-forward ERROR: {type(e).__name__}: {e}")
        wf_result = None

    try:
        checks = run_stop_ship_checks(
            best_metrics=best_metrics, best_objective=best_obj,
            walkforward_result=wf_result, stress_report=best_stress,
            dataset_audit=audit,
            validation_result=validation_result,
            holdout_report=holdout_report,
            sensitivity_penalty=sensitivity_penalty,
            recent_window_result=recent_window_result,
            parity_result=parity_result,
            cluster_report=cluster_report,
            long_parity_result=long_parity_result,
            execution_realism={
                "connector": connector,
                "taker_probability": taker_prob,
                "supports_post_only": pair_rules.supports_post_only,
            },
        )
        mandatory_gates_failed = [
            name for name in MANDATORY_GATES if checks.get(name) is False
        ]
        if validation_errors:
            validation_status = "validation_error"
        elif mandatory_gates_failed:
            validation_status = "validated_fail"
        else:
            validation_status = "validated_pass"
    except Exception as e:
        validation_errors.append(("stop_ship_checks", type(e).__name__, str(e)))
        validation_status = "validation_error"
        print(f"  Stop-ship checks error: {e}")

    # Move YAML based on outcome
    import shutil as _shutil
    if pending_yaml_path and Path(pending_yaml_path).exists():
        if validation_status == "validated_pass":
            yaml_path = str(_out_dir / _yaml_filename)
            _shutil.move(pending_yaml_path, yaml_path)
        else:
            _rejected_dir = _out_dir / "rejected"
            _rejected_dir.mkdir(parents=True, exist_ok=True)
            yaml_path = str(_rejected_dir / _yaml_filename)
            _shutil.move(pending_yaml_path, yaml_path)
            # Drop a REJECTED.json sibling marker
            import json as _json
            _marker = Path(yaml_path).with_suffix("").as_posix() + "_REJECTED.json"
            Path(_marker).write_text(_json.dumps({
                "validation_status": validation_status,
                "mandatory_gates_failed": mandatory_gates_failed,
                "validation_errors": [
                    {"step": step, "type": t, "message": m}
                    for step, t, m in validation_errors
                ],
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "dataset_hash": dataset_hash,
                "mandatory_gates_policy": {
                    "frozen_parity_policy": FROZEN_PARITY_POLICY,
                    "mandatory_gates": sorted(list(MANDATORY_GATES)),
                },
            }, indent=2))

    result_entry["status"] = validation_status
    result_entry["validation_status"] = validation_status
    result_entry["validation_errors"] = validation_errors
    result_entry["mandatory_gates_failed"] = mandatory_gates_failed
    result_entry["yaml_path"] = yaml_path
    result_entry["checks"] = checks

    try:
        _run_provenance = {
            "notebook": "direction-custom/mr_bb_rsi",
            "run_timestamp": datetime.now(timezone.utc).isoformat(),
            "n_jobs": N_JOBS,
            "objective_version": OBJECTIVE_VERSION,
            "search_controller_compat": SEARCH_CONTROLLER_COMPAT,
            "validation_controller_compat": VALIDATION_CONTROLLER_COMPAT,
            "refresh_close_mode": REFRESH_CLOSE_MODE,
            "initial_base_balance": INITIAL_BASE_BALANCE,
            "taker_probability": taker_prob,
            "trial_number": best["trial_number"],
            "validation_status": validation_status,
        }
        generate_report(
            study_name=study_name,
            dataset_summary={
                "connector": connector, "trading_pair": pair, "interval": interval,
                "n_candles": len(candles), "dataset_hash": dataset_hash,
                "n_trials_phase1": N_TRIALS, "n_candidates_stressed": len(top_candidates),
                "total_amount_quote_search_min": 50.0,
                "total_amount_quote_search_max": 500.0,
                "total_amount_quote_ideal": best_engine_config.total_amount_quote,
                "search_controller_compat": SEARCH_CONTROLLER_COMPAT,
            },
            best_params=best["params"], best_metrics=best_metrics, best_objective=best_obj,
            walkforward_result=wf_result, stress_report=best_stress,
            stop_ship_checks=checks,
            holdout_report=holdout_report,
            dataset_audit=audit,
            sensitivity_report=sensitivity_report,
            recent_window_result=recent_window_result,
            recent_window_results=recent_window_results,
            recent_blocking_window_days=RECENT_BLOCKING_WINDOW_DAYS,
            cluster_report=cluster_report,
            yaml_validation_result=validation_result,
            dataset_slices=dataset_slices,
            parity_result=parity_result,
            long_parity_result=long_parity_result,
            run_provenance=_run_provenance,
            execution_realism={
                "taker_probability": taker_prob,
                "supports_post_only": pair_rules.supports_post_only,
                "connector": connector,
                "fill_participation_rate": 0.1,
                "latency_bars": 1,
                "slippage_bps": 5.0,
                "refresh_close_mode": REFRESH_CLOSE_MODE,
            },
            tp_min_notional_failures=getattr(best_metrics, "tp_min_notional_failures", 0),
            output_path=f"artifacts/direction-custom/mr_bb_rsi/{connector}/{pair.replace('-', '_').lower()}_{interval}_report.md",
        )
        _gates_pass = sum(1 for v in checks.values() if v)
        _gates_total = len(checks)
        result_entry["gates_pass"] = _gates_pass
        result_entry["gates_total"] = _gates_total
        _total_time = time.time() - pair_start
        print(f"  Total time: ({_total_time/60:.1f}min)  Gates: {_gates_pass}/{_gates_total}  Status: {validation_status}")
        if yaml_path:
            print(f"  YAML: {yaml_path}")
        if mandatory_gates_failed:
            print(f"  Failed mandatory gates: {mandatory_gates_failed}")
    except Exception as e:
        print(f"  Report error: {e}")

    _pair_bar.update(1)

_pair_bar.close()

total_elapsed = time.time() - sweep_start
print(f"\n{'='*60}")
print(f"SWEEP COMPLETE: {len(candidates)} connector/pair combinations in {total_elapsed/60:.1f} minutes")
print(f"{'='*60}")


## 4. Results Summary

In [ ]:
# Results summary — status counts, per-pair outcomes, and compact sorted table.
import os
from pathlib import Path

def _status_counts(rows):
    counts = {}
    for r in rows:
        counts[r.get("status", "?")] = counts.get(r.get("status", "?"), 0) + 1
    return counts

print("=" * 60)
print("SWEEP RESULTS SUMMARY")
print("=" * 60)
print("Status counts:", _status_counts(sweep_results))

print("\nPer-pair outcomes:")
for r in sweep_results:
    status = r.get("validation_status", r.get("status", "?"))
    conn = r.get("connector", "?")
    pair = r.get("pair", r.get("trading_pair", "?"))
    extras = ""
    if status in ("validated_pass", "complete"):
        extras = f" score={r.get('robust_score', r.get('best_score', 0)):.3f}  yaml={r.get('yaml_path')}"
    elif status == "validated_fail":
        failed = r.get("mandatory_gates_failed", [])
        extras = f" failed_gates={failed}  yaml={r.get('yaml_path')}"
    elif "reason" in r:
        extras = f" reason={r['reason']}"
    elif "error" in r:
        extras = f" error={str(r['error'])[:80]}"
    print(f"  [{status:20s}] {conn:8s} {pair:15s}{extras}")


# ── Compact sorted results table (ML-DIR-001, ML-DIR-003) ──
# Primary: validation_status == validated_pass (or legacy "complete").
# Secondary: validation_status == validated_fail (rejected candidates).
_primary = [
    r for r in sweep_results
    if r.get("validation_status", r.get("status")) in ("validated_pass", "complete")
]
_rejected = [
    r for r in sweep_results
    if r.get("validation_status", r.get("status")) == "validated_fail"
]
_primary_sorted = sorted(
    _primary,
    key=lambda r: r.get("robust_score", float("-inf")) if r.get("robust_score") is not None else float("-inf"),
    reverse=True,
)
# Keep legacy name for back-compat with test fixtures that reference it
_completed_sorted = _primary_sorted

print("\n" + "=" * 100)
print("COMPACT RESULTS TABLE (sorted by robust_score descending)")
print("=" * 100)

_header = f"{'Rank':>4}  {'Connector':<10}  {'Pair':<18}  {'Robust':>8}  {'Holdout':>8}  {'Recent28d':>10}  {'Gates':>7}  {'DataDays':>8}  {'YAML'}"
print(_header)
print("-" * 100)

for _rank, _r in enumerate(_completed_sorted, start=1):
    _conn = _r.get("connector", "?")
    _pair = _r.get("pair", _r.get("trading_pair", "?"))
    _robust = _r.get("robust_score")
    _robust_s = f"{_robust:>8.4f}" if isinstance(_robust, (int, float)) else f"{'N/A':>8}"
    _hr = _r.get("holdout_report")
    if _hr is not None:
        _hs = getattr(_hr, "exported_holdout_score", None)
        _holdout_s = f"{_hs:>8.4f}" if isinstance(_hs, (int, float)) else f"{'N/A':>8}"
    else:
        _holdout_s = f"{'N/A':>8}"
    _rw = _r.get("recent_window_result")
    if _rw is not None and getattr(_rw, "objective", None) is not None:
        _rs = getattr(_rw.objective, "raw_score", None)
        _recent_s = f"{_rs:>10.4f}" if isinstance(_rs, (int, float)) else f"{'N/A':>10}"
    else:
        _recent_s = f"{'N/A':>10}"
    _checks = _r.get("checks") or {}
    _gp = sum(1 for v in _checks.values() if v)
    _gt = len(_checks) if _checks else 0
    _gates_s = f"{_gp}/{_gt}" if _gt else "N/A"
    _dd = _r.get("dataset_days")
    _datadays_s = f"{_dd:>6.0f}d" if isinstance(_dd, (int, float)) else f"{'N/A':>8}"
    _yaml = _r.get("yaml_path") or "-"
    _yaml_s = os.path.basename(_yaml) if _yaml != "-" else "-"
    print(f"{_rank:>4}  {_conn:<10}  {_pair:<18}  {_robust_s}  {_holdout_s}  {_recent_s}  {_gates_s:>7}  {_datadays_s:>8}  {_yaml_s}")

if not _completed_sorted:
    print("  (no validated pairs)")
print("=" * 100)

# Secondary: rejected candidates (validated_fail) with their failed gates.
if _rejected:
    print("\n" + "=" * 100)
    print(f"REJECTED CANDIDATES ({len(_rejected)}) — YAML under rejected/ subdir")
    print("=" * 100)
    for _r in _rejected:
        _conn = _r.get("connector", "?")
        _pair = _r.get("pair", _r.get("trading_pair", "?"))
        _failed = _r.get("mandatory_gates_failed", [])
        _yaml = _r.get("yaml_path") or "-"
        _yaml_s = os.path.basename(_yaml) if _yaml != "-" else "-"
        print(f"  {_conn:<10}  {_pair:<18}  failed_gates={_failed}  yaml={_yaml_s}")
    print("=" * 100)


## 5. Profitable Pairs Detail by Exchange

In [ ]:
# Profitable candidates — only those that passed validation gates (ML-DIR-001)
profitable = [
    r for r in sweep_results
    if r.get("validation_status", r.get("status")) in ("validated_pass", "complete")
    and r.get("robust_score", 0.0) is not None
    and r.get("robust_score", 0.0) > 0
]
print(f"\n{'='*60}")
print(f"Profitable & validated pairs: {len(profitable)}")
print(f"{'='*60}")

# Informational release gates. None are blocking (already enforced in the pipeline).
print("\nRelease Gates (Informational Only):")
for r in profitable:
    pair = r.get("pair", r.get("trading_pair", "?"))
    print(f"\n  {r['connector']} / {pair}:")
    rs = r.get("robust_score", 0.0)
    brf = r.get("total_reject_fraction")
    gates = [
        ("robust_score > 0", rs, 0.0, rs is not None and rs > 0),
    ]
    if brf is not None:
        gates.append(
            ("order_reject_fraction < 0.30", brf, 0.30, brf < 0.30),
        )
    for name, actual, threshold, passed in gates:
        mark = "PASS" if passed else "FAIL"
        print(f"    [{mark}] {name}: actual={actual}")


## 6. Cross-Pair Ranking

In [ ]:
# ── CROSS-PAIR RANKING (ML-DIR-001, ML-DIR-003) ──
validated = [
    r for r in sweep_results
    if r.get("validation_status", r.get("status")) in ("validated_pass", "complete")
]
rejected = [
    r for r in sweep_results
    if r.get("validation_status", r.get("status")) == "validated_fail"
]

def _rs(r):
    v = r.get("robust_score")
    return v if isinstance(v, (int, float)) else float("-inf")

validated.sort(key=_rs, reverse=True)

print(f"\n{'='*60}")
print(f"Cross-pair ranking by robust_score (best first)")
print(f"{'='*60}")
for rank, r in enumerate(validated, 1):
    pair = r.get("pair", r.get("trading_pair", "?"))
    rs = r.get("robust_score", 0.0)
    rs_s = f"{rs:>8.3f}" if isinstance(rs, (int, float)) else f"{'N/A':>8}"
    print(f"  #{rank:>2} {r['connector']:8s} {pair:15s} "
          f"robust_score={rs_s}  "
          f"yaml={r.get('yaml_path')}")

if rejected:
    print(f"\n{'='*60}")
    print(f"Rejected candidates ({len(rejected)}) — YAML in rejected/ subdir")
    print(f"{'='*60}")
    for r in rejected:
        pair = r.get("pair", r.get("trading_pair", "?"))
        failed = r.get("mandatory_gates_failed", [])
        print(f"  {r['connector']:8s} {pair:15s} failed_gates={failed}")


## 7. Next Steps

- Open each exported YAML in `artifacts/direction-custom/mr_bb_rsi/<connector>/`
  and confirm the parameters look reasonable for live deployment.
- Run the multi-exchange sweep first to discover candidate pairs, then
  populate `RETEST_PAIRS` here for a focused run.
- The cross-pair ranking in cell 14 lets you compare the same strategy
  across pairs on the same scale.
